In [ ]:
import pandas as pd
import pickle as pkl
import os
from parallel_simplify import run_parallel_simplification_w_llm, load_simplified_results

In [ ]:
with open('OpenAI_token.txt', 'r') as file:
    content = file.read()
    os.environ["OPENAI_API_KEY"] = content

with open('Gemini_token.txt', 'r') as file:
    content = file.read()
    os.environ["GEMINI_API_KEY"] = content

In [ ]:
df_to_simplify = pd.read_json(os.path.join('data', 'simpletext25_task11_test.json'))

with open('prepared_statements.pkl', 'rb') as f: # you need to have constructed the prepared_statements.pkl file containing the original sentences with marked complex phrases
    df_to_simplify['prep_snt'] = pkl.load(f)

In [ ]:
df_to_simplify

In [ ]:
new_simp_prompt = '''You are a text simplification system with good global and language knowledge but no expertise in specific domains.
You will be given 10 texts, TREAT THEM SEPERATLY and also return 10 simplified texts.
Complex domain specific or scientific terminology is marked by square brackets (e.g., [primary acoustic modeling]).
For each text, replace or explain the terms in square brackets in order to make it more understandable for non-experts and simplify for non-expert readers.
Make sure to exactly return an enumerated list of 10 simplified texts and that all terms in square brackets are simplified or explained! 
Do not simplify the structure of the texts or complex language independent of the difficult scientific descriptions. RETURN ONLY THE 10 TEXTS.

TEXTS: 
'''

new_simp_prompt_single = '''You are a text simplification system with good global and language knowledge but no expertise in specific domains.
You will be given 1 text.
Complex domain specific or scientific terminology is marked by square brackets (e.g., [primary acoustic modeling]).
For each text, replace or explain the terms in square brackets in order to make it more understandable for non-experts and simplify for non-expert readers.
Make sure to exactly return an enumerated list of 1 simplified text and that all terms in square brackets are simplified or explained! 
Do not simplify the structure of the texts or complex language independent of the difficult scientific descriptions. RETURN ONLY THE 1 TEXT.

TEXT: 
'''

comp = '''You are a text complication system with good global and language knowledge but and expertise in specific domains.
You will be given 10 texts, TREAT THEM SEPERATLY and also return 10 more complex texts.
Please make the texts more complex (not in their structure) but not considerably longer. RETURN ONLY THE 10 TEXTS.

TEXTS: 
'''

comp_single = '''You are a text complication system with good global and language knowledge but and expertise in specific domains.
You will be given 1 text.
Please make the text more complex (not in the structure) but not considerably longer. RETURN ONLY THE 1 TEXT.

TEXTS: 
'''

rephrase = '''You are a text rephrasing system.
You will be given 10 texts, TREAT THEM SEPERATLY and also return 10 texts.
Please rephrase the texts. Do not change their level of complexity (so do not make them more difficult in their structure) and do not make them considerably longer. 
RETURN ONLY THE 10 TEXTS as a list.

TEXTS:  
'''

rephrase_single = '''You are a text rephrasing system.
You will be given 1 texts.
Please rephrase the text. Do not change its level of complexity (so do not make it more difficult in its structure) and do not make it considerably longer. 
RETURN ONLY THE 1 TEXT.

TEXT: 
'''

start2_step_no_topic_CTI_pre_no_examples_list_deepl = '''You are given 10 texts, TREAT THEM SEPARATELY.
Complex technical or scientific terms are indicated by square brackets (e.g. [convolutional neural network]).
For each text, replace or explain the terms in square brackets to make it understandable to non-experts and to simplify for non-expert readers.
Make sure that you return exactly one enumerated list of 10 simplified texts and that all terms in square brackets are simplified or explained!

'''

start2_step_no_topic_CTI_pre_no_examples_list_deepl_single = '''You are given 1 text.
Complex technical or scientific terms are indicated by square brackets (e.g. [convolutional neural network]).
For each text, replace or explain the terms in square brackets to make it understandable to non-experts and to simplify for non-expert readers.
Make sure that you return exactly one enumerated list of 1 simplified text and that all terms in square brackets are simplified or explained!

'''


In [ ]:
nils_1 = '''You are at a conference as an expert explaining the contents of these texts to non-experts.
There will be 10 different texts, treat each of them separately and only return the simplified version as a python list.
Complex terms or domain-specific terminology will be marked with square brackets (e.g., [primary acoustic modeling]).
To make the subject more understandable to these non-experts, you will replace or explain the terms marked by square brackets. 
Do not simplify the structure or texts in any other way than specified, or the non-experts will lose interest.

'''

nils_1_single = '''You are at a conference as an expert explaining the contents of these texts to non-experts.
There will be 1 text.
Complex terms or domain-specific terminology will be marked with square brackets (e.g., [primary acoustic modeling]).
To make the subject more understandable to these non-experts, you will replace or explain the terms marked by square brackets. 
Do not simplify the structure or texts in any other way than specified, or the non-experts will lose interest.

'''

ide_2 = ''' Think of yourself as a translator: You're translating complex, technical language (found inside [square brackets]) into everyday language.
The Job: You will be given 10 documents all at the same time.
Your Task for Each Document:
    Find all [bracketed phrases].
    "Translate" them into simple terms for a general audience.
    Important: Do not "translate" or change anything outside the brackets. Keep the original sentence flow. Result Expected: A list of 10 translated documents, ensuring all original bracketed items have been simplified. Do not output the original input texts.

'''

ide_2_single = ''''Think of yourself as a translator: You're translating complex, technical language (found inside [square brackets]) into everyday language.
The Job: You will be given 1 document.
Your Task:
    Find all [bracketed phrases].
    "Translate" them into simple terms for a general audience.
    Important: Do not "translate" or change anything outside the brackets. Keep the original sentence flow. Result Expected: A translated document, ensuring all original bracketed items have been simplified. Do not output the original input texts.

'''

nico_1 = '''Hello! we have been given an important task, which i heard you can complete ecxeptionally well, as you are the best and most reliable system for text simplification and language in general.
we have been given 10 texts which should be simplified or explained. problematic, complex words are marked within [square brackets]
Please, for each of these 10 texts, replace or explain these [terms] and return an enumerated list of 10 simplified texts
please do not change the texts structure and please ONLY return these 10 texts! thank you!
here are the said 10 texts:
'''


In [ ]:
prompts = {
    'p1' : (new_simp_prompt, new_simp_prompt_single),
    'p2' : (start2_step_no_topic_CTI_pre_no_examples_list_deepl, start2_step_no_topic_CTI_pre_no_examples_list_deepl_single),
    'c' : (comp, comp_single),
    'r' : (rephrase, rephrase_single),
    'pn1' : (nils_1, nils_1_single),
    'pi2' : (ide_2, ide_2_single),
    'pni1' : (nico_1, new_simp_prompt_single)
}
used_llms = {'True' : 'ChatGPT', 'False' : 'Gemini'}

In [ ]:
def save_simplified_output(df, results, RUN_ID, output_path="simplified_output.tsv"):
    results.sort(key=lambda x: int(x[0]))
    ids, simplified_sents = zip(*results)
    output_df = df.loc[list(ids)].copy()
    output_df = output_df.drop('prep_snt', axis=1)
    output_df["prediction"] = simplified_sents
    output_df['run_id'] = RUN_ID
    output_df.to_csv(output_path + '.tsv', sep="\t", index_label="id")
    output_df.to_json(output_path + '.json', orient="records", index=False)    

def fix_simplified_output(original_df, tsv_output, json_output, run_id):
    original_df['prediction'] = original_df['complex']
    original_df['run_id'] = run_id
    
    tsv_output=pd.read_csv(tsv_output, sep='\t', index_col="id")    
    common_columns = original_df.columns.intersection(tsv_output.columns)

    merged_df = tsv_output.combine_first(original_df[common_columns])
    merged_df.to_json(json_output, orient="records", index=False)    

In [ ]:
df_to_simplify

In [ ]:
run_chatgpt = False # todo: modify
modelname = 'gemini-2.0-flash' #'gemini-2.5-flash-preview-05-20' # 'gemini-2.0-flash' # 'gpt-4.1-nano' # todo: modify

In [ ]:
for prompt_id in ['p1', 'p2', 'p3']: # todo: modify
        
    RUN_ID = 'THM_task11_' + prompt_id + '--' + modelname
    OUTPUT_FILE = 'simp__' + RUN_ID

    run_parallel_simplification_w_llm(df_to_simplify, RUN_ID, prompts[prompt_id], run_chatgpt, modelname, parts = len(df_to_simplify)/10)

    output_dir = f"./simplified_chunks/{RUN_ID}"
    if not os.path.exists(output_dir):
        print("Output directory not found!")
        
    results = load_simplified_results(RUN_ID)
    save_simplified_output(df_to_simplify, results, output_path=OUTPUT_FILE, RUN_ID=RUN_ID)

    print(f"✅ Total simplified sentences loaded: {len(results)}, run completely done, results at: {OUTPUT_FILE}.json")
    
    fix_simplified_output(df_to_simplify, 'simp__' + RUN_ID + '.tsv', 'simp_f_' + RUN_ID + '.json', RUN_ID)